# Solar Filament Segmentation — Kaggle runnerThis notebook holds **no logic**. It clones the pipeline from GitHub at a pinnedrevision, installs what the Kaggle image is missing, and calls the same CLIentry points used locally. Every code change is made in the repository; the onlything edited here is the revision below.Requirements: *Internet* enabled in the notebook settings (Settings -> Internet),and a GPU accelerator (T4 x2 or P100) for training.

In [ ]:
# --- the only cell you normally edit -----------------------------------------
REPO_URL = "https://github.com/ShreyPatel1311/solar-filament-segmentation.git"
REVISION = "main"          # branch, tag, or full commit SHA - pin a SHA for a final run
CONFIG   = "configs/unet_resnet34.yaml"
OVERRIDES = []             # e.g. ["train.epochs=25", "data.image_size=768"]
# -----------------------------------------------------------------------------

In [ ]:
import os, subprocess, sys, shutil, pathlib

WORK_DIR = pathlib.Path("/kaggle/working")
os.chdir(WORK_DIR)  # always stand outside REPO_DIR before touching it, so a
                     # re-run of this cell (no kernel restart) can't delete the
                     # directory the process is currently sitting in

REPO_DIR = WORK_DIR / "repo"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always start from a clean checkout

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REVISION], check=True)

commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                        check=True, capture_output=True, text=True).stdout.strip()
print("running commit", commit)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
os.environ["PYTHONPATH"] = str(REPO_DIR / "src")  # so the ! scripts below import filseg too

In [ ]:
# Kaggle already ships torch, numpy, opencv, scikit-image, pandas and pillow.
# Only the missing pieces are installed, and the versions live in the repo so a
# dependency fix is a commit rather than a notebook edit.
import os

os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"   # skip albumentations' update ping

# Resolved install: pure-Python / CPU dependencies only, nothing CUDA.
!pip install -q -r requirements-kaggle.txt
# --no-deps: keeps pip from swapping the image's preinstalled CUDA torch build.
!pip install -q --no-deps -r requirements-kaggle-nodeps.txt

In [ ]:
# Fail here, loudly, rather than three cells later inside a training run.
import albucore, albumentations, cv2, pycocotools, segmentation_models_pytorch as smp, torch

import filseg
from filseg.data.transforms import val_transforms  # the import that needs albucore to match
from filseg.paths import resolve_paths

print("filseg       ", filseg.__version__)
print("torch        ", torch.__version__, "| cuda:", torch.cuda.is_available())
print("albumentations", albumentations.__version__, "| albucore", albucore.__version__)
print("smp          ", smp.__version__, "| opencv", cv2.__version__)

paths = resolve_paths()
print("data  ", paths.data_root)
print("train ", len(list(paths.train_images.glob("*.jpeg"))), "images")
print("test  ", len(list(paths.test_images.glob("*.jpeg"))), "images")

## Train

In [ ]:
args = ["--config", CONFIG] + [a for o in OVERRIDES for a in ("--set", o)]
!python scripts/train.py {" ".join(args)}

## Validate with the leaderboard metric

In [ ]:
CHECKPOINT = "/kaggle/working/checkpoints/unet_r34_best.pt"
!python scripts/evaluate.py --checkpoint {CHECKPOINT}

## Predict the test set and write `submission.csv`

In [ ]:
!python scripts/predict.py --checkpoint {CHECKPOINT} --out /kaggle/working/submission.csv

In [ ]:
import pandas as pd

submission = pd.read_csv("/kaggle/working/submission.csv")
print(submission.shape, "rows |", submission.filament_id.str.rsplit("_", n=1).str[0].nunique(), "images")
submission.head()